# Emotion Music Studio — free GPU backend (Colab)

Runs the existing FastAPI backend with **`musicgen-medium`** on Colab's free **T4 GPU** and exposes a public URL via cloudflared.

**First:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

Open this notebook directly in Colab:
`https://colab.research.google.com/github/AthSri0507/Multi_Modal-Music-Generation/blob/main/deploy/colab_gpu_backend.ipynb`

In [ ]:
# 1. Clone your repo
REPO_URL = "https://github.com/AthSri0507/Multi_Modal-Music-Generation.git"
import os
if not os.path.exists('repo'):
    !git clone --depth 1 $REPO_URL repo
%cd repo

In [ ]:
# 2. Install serving deps. Colab already ships a CUDA build of torch, so do NOT
#    reinstall torch (that would downgrade it / break CUDA).
!pip install -q transformers sentencepiece protobuf soundfile fastapi uvicorn python-multipart librosa
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — set Runtime to T4 GPU')

In [ ]:
# 3. (optional) build the React UI so the backend serves it at /
import shutil
if shutil.which('node'):
    !cd frontend && npm install --silent && npm run build
else:
    print('node not found; API-only. You can still call the API or run the React dev server locally against the tunnel URL.')

In [ ]:
# 4. Configure the bigger model + CLAP ranking, then launch uvicorn in the background
import os, subprocess, time
os.environ['MUSICGEN_MODEL_NAME'] = 'facebook/musicgen-medium'  # GPU handles this easily
os.environ['MUSICGEN_USE_CLAP'] = '1'                            # Stage-2 CLAP ranking
os.environ['PYTHONPATH'] = os.getcwd()
log = open('server.log', 'w')
server = subprocess.Popen(['uvicorn', 'src.api.app:app', '--host', '0.0.0.0', '--port', '8000'],
                          stdout=log, stderr=subprocess.STDOUT)
time.sleep(8)
print('server starting on :8000 (MusicGen loads lazily on the first generate; first call downloads ~3.5GB for medium)')

In [ ]:
# 5. Public URL via cloudflared (no account needed)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import subprocess, re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    print(line.strip())
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('\n==== PUBLIC URL:', url, '====\nOpen it in a browser (serves the UI if built), or use it as VITE_API_BASE for the local React app.')

In [ ]:
# 6. Quick test (run after the URL prints). First generate downloads the model, so be patient.
import requests
print(requests.get(url + '/api/v1/health').json())  # expect device: cuda
r = requests.post(url + '/api/v1/music/generate', json={'prompt': 'happy upbeat jazz trio', 'duration': 10, 'preset': 'balanced'})
print('generated:', r.json().get('id'), '| mood', r.json().get('mood'), '| genre', r.json().get('genre'))

**Notes**
- The gallery (SQLite) + audio live on the Colab disk and reset when the runtime stops. Mount Google Drive to persist.
- If `device` is `cpu`, set Runtime → Change runtime type → **T4 GPU** and re-run.
- The free tunnel URL changes each session.